<a href="https://colab.research.google.com/github/NikoriakViktot/PY-Course-Victor-Nikoriak-22-09-2026/blob/main/module_2/lessons/lesson_19_classes_namespace/note_lesson_19_classes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Урок 19 — Класи, простір імен: сервіс доставки тримає свої правила

У фінальному проєкті модуля 1 дані (`orders`) жили окремо від функцій, і правила — «номери видає лише сервіс», «одна доставка на замовлення» — можна було оминути, просто дописавши в список. Сьогодні стан і правила переїжджають в один **клас**.

Виконуй клітинки **зверху вниз**; перед **Прогнозом** спершу скажи, що буде. Теорія, схеми архітектури й порівняння рішень — у книзі: [Урок 19. Класи, простір імен](https://nikoriakviktot.github.io/PY-Course-Victor-Nikoriak-22-09-2026/modules/m2/lesson_19/).

## 🔁 Пригадай (без підглядання)

1. Що надрукує `b = a; b.append(4); print(a)` для `a = [1, 2, 3]`?
2. Що зберігає замикання `next_id` з `make_counter()` уроку 18?
3. Чим `NamedTuple` відрізняється від словника?

<details>
<summary>Відповіді</summary>

1. `[1, 2, 3, 4]` — два ярлики одного об'єкта.
2. Змінну `count` оточуючої функції.
3. Поля фіксовані, доступ через крапку, кортеж незмінний.

</details>

## 1. Усе — об'єкти

**Прогноз:** який тип у `int`?

In [ ]:
print(type(540), type("Поділ"), type([540, 320]))
print(isinstance(540, object), isinstance(len, object))
print(type(int), type(str))

<details>
<summary>Відповідь</summary>

`<class 'type'>`: клас — теж об'єкт, і його клас — `type`.

</details>

## 2. Перший клас `Order`

In [ ]:
from datetime import datetime


class Order:
    """Замовлення кафе."""

    def __init__(self, order_id, time, bill, tip, guests):
        self.id = order_id
        self.time = time
        self.bill = bill
        self.tip = tip
        self.guests = guests

    def total(self):
        """Чек разом з чайовими."""
        return self.bill + self.tip

    def per_guest(self):
        return round(self.bill / self.guests, 2)


order = Order(1, datetime(2024, 7, 19, 18, 30), 540.0, 50.0, 2)
print(order.bill, order.total(), order.per_guest())

**Прогноз:** чи однакові `Order.total(order)` і `order.total()`?

In [ ]:
print(Order.total(order) == order.total())

<details>
<summary>Відповідь</summary>

Так: `self` — це сам об'єкт; Python передає його першим аргументом.

</details>

### `__repr__`

Без нього `print(order)` покаже адресу в пам'яті. З ним — зрозумілий рядок.

In [ ]:
class Order:
    """Замовлення кафе."""

    def __init__(self, order_id, time, bill, tip, guests):
        self.id = order_id
        self.time = time
        self.bill = bill
        self.tip = tip
        self.guests = guests

    def __repr__(self):
        return f"Order(№{self.id}, {self.time:%Y-%m-%d %H:%M}, {self.bill:.2f} грн)"

    def total(self):
        return self.bill + self.tip


orders = [Order(1, datetime(2024, 7, 19, 18, 30), 540.0, 50.0, 2),
          Order(2, datetime(2024, 7, 19, 12, 10), 320.0, 30.0, 1)]
print(orders[0])
print(orders)

## 🛠 Вправа 1. Клас `Customer`

Атрибути `name`, `phone`, `orders` (**власний** порожній список для кожного клієнта); метод `add(order)`; метод `spent()` — сума `bill` усіх замовлень; `__repr__` → `Customer(Оксана, замовлень: 2)`.

In [ ]:
class Customer:
    # YOUR CODE HERE
    # BEGIN SOLUTION
    def __init__(self, name, phone):
        self.name = name
        self.phone = phone
        self.orders = []

    def __repr__(self):
        return f"Customer({self.name}, замовлень: {len(self.orders)})"

    def add(self, order):
        self.orders.append(order)

    def spent(self):
        return sum(order.bill for order in self.orders)
    # END SOLUTION


oksana = Customer("Оксана", "050-123-4567")
taras = Customer("Тарас", "097-987-6543")
oksana.add(orders[0])
oksana.add(orders[1])
print(oksana, taras)
assert repr(oksana) == "Customer(Оксана, замовлень: 2)"
assert oksana.spent() == 860.0 and taras.orders == [], "у кожного клієнта — свій список"
print("✅ Вправа 1 пройдена")

## 3. Атрибути класу й простір імен

In [ ]:
class Delivery:
    FEES = {"Поділ": 60, "Оболонь": 80, "Печерськ": 90}

    def __init__(self, order_id, district, driver):
        self.order_id = order_id
        self.district = district
        self.driver = driver

    def fare(self):
        return self.FEES[self.district]


first = Delivery(1, "Оболонь", "D-3")
second = Delivery(2, "Поділ", "D-1")
print(first.fare(), second.fare())
print(first.FEES is second.FEES is Delivery.FEES)


print(first.__dict__)
print("FEES" in Delivery.__dict__, "FEES" in first.__dict__)

### Пастка: присвоєння через `self`

**Прогноз:** що надрукує `print(courier.trips, Courier.trips)` після двох `deliver()`?

In [ ]:
class Courier:
    trips = 0

    def __init__(self, name):
        self.name = name

    def deliver(self):
        self.trips += 1


courier = Courier("D-3")
courier.deliver()
courier.deliver()
print(courier.trips, Courier.trips)
print(courier.__dict__)

<details>
<summary>Відповідь</summary>

`2 0`: присвоєння `self.trips = …` створило атрибут в **екземплярі**, який затінив класовий. Щоб рахувати спільно — `Courier.trips += 1`.

</details>

### 🐛 Вправа 2. Спільний кошик

**Прогноз:** що в кошику Тараса? Потім виправ клас.

In [ ]:
class Cart:
    items = []                       # ← у чому проблема?

    def add(self, dish):
        self.items.append(dish)
# BEGIN SOLUTION
class Cart:
    def __init__(self):
        self.items = []

    def add(self, dish):
        self.items.append(dish)
# END SOLUTION


oksana_cart, taras_cart = Cart(), Cart()
oksana_cart.add("борщ")
print(taras_cart.items)
assert taras_cart.items == [], "кошик Тараса має бути порожнім"
print("✅ Вправа 2 пройдена")

### Клас — не оточення для методів

Розкоментуй другий рядок `return` і запусти: `NameError: name 'VAT' is not defined`. Методи шукають імена за LEGB, а простір імен класу туди не входить.

In [ ]:
class Menu:
    VAT = 0.2

    def price_with_vat(self, price):
        return price * (1 + self.VAT)
        # return price * (1 + VAT)


print(Menu().price_with_vat(100))

## 4. `@classmethod` і `@staticmethod`

Перевірки з уроку 13 тепер у `__init__`: неправильне замовлення неможливо створити.

In [ ]:
class Order:
    """Замовлення кафе."""

    def __init__(self, order_id, time, bill, tip, guests):
        if bill <= 0:
            raise ValueError(f"сума чека має бути більшою за 0, а маємо {bill}")
        if guests < 1:
            raise ValueError(f"гостей має бути хоча б один, а маємо {guests}")
        self.id = order_id
        self.time = time
        self.bill = bill
        self.tip = tip
        self.guests = guests

    def __repr__(self):
        return f"Order(№{self.id}, {self.time:%Y-%m-%d %H:%M}, {self.bill:.2f} грн)"

    @classmethod
    def from_line(cls, line, order_id):
        """Рядок каси "2024-07-19 18:30;540.00;50;2" → Order."""
        fields = line.split(";")
        if len(fields) != 4:
            raise ValueError(f"очікували 4 поля, а маємо {len(fields)}")
        time_text, bill, tip, guests = fields
        return cls(order_id, datetime.strptime(time_text, "%Y-%m-%d %H:%M"),
                   float(bill), float(tip), int(guests))

    @staticmethod
    def meal_type(hour):
        if 11 <= hour <= 15:
            return "обід"
        if 17 <= hour <= 23:
            return "вечеря"
        return "інше"

    def meal(self):
        return self.meal_type(self.time.hour)


order = Order.from_line("2024-07-21 21:30;1200.00;150;6", 7)
print(order, order.meal())
print(Order.meal_type(12))

In [ ]:
try:
    Order.from_line("2024-07-21 14:20;610.00;60;0", 8)
except ValueError as error:
    print(error)

## 5. `isinstance` і `type`

**Прогноз:** чи є `True` цілим числом?

In [ ]:
print(type(order) is Order, isinstance(order, Order))
print(isinstance(True, int), type(True) is int)

<details>
<summary>Відповідь</summary>

`isinstance(True, int)` — `True`: `bool` — нащадок `int`. Але `type(True) is int` — `False`.

</details>

## 6. Архітектура: три способи тримати стан

| | А. Функції над даними | Б. Замикання | В. Клас |
|---|---|---|---|
| Де стан | списки, що передають у функції | змінні фабрики | атрибути об'єкта |
| Хто стежить за правилами | кожен, хто викликає | функції фабрики | методи класу |
| Коли доречно | скрипт, разовий звіт | маленький стан, 1–2 дії | стан + багато дій + правила |

Шари системи: **інтерфейс** (`app.py`) → **сервіс** (`DeliveryService`) → **моделі** (`Order`, `Delivery`) і **сховище** (`data.json`). Стрілки — в один бік: моделі нічого не знають про `print` і `sys.argv`. Схеми — у книзі.

## 7. Розібраний приклад: `DeliveryService`

In [ ]:
class DeliveryService:
    def __init__(self):
        self.orders = {}
        self.deliveries = {}
        self.next_id = 1

    def add_order(self, line):
        order = Order.from_line(line, self.next_id)
        self.orders[order.id] = order
        self.next_id += 1
        return order

    def add_delivery(self, order_id, district, driver):
        if order_id not in self.orders:
            raise ValueError(f"замовлення №{order_id} немає")
        if order_id in self.deliveries:
            raise ValueError(f"замовлення №{order_id} вже має доставку")
        delivery = Delivery(order_id, district, driver)
        self.deliveries[order_id] = delivery
        return delivery

    def report(self):
        cafe = sum(order.bill for order in self.orders.values())
        taxi = sum(delivery.fare() for delivery in self.deliveries.values())
        return (f"Замовлень: {len(self.orders)}, з доставкою: {len(self.deliveries)}; "
                f"кафе {cafe:.2f} грн, таксі {taxi} грн")


smachno = DeliveryService()
smachno.add_order("2024-07-19 18:30;540.00;50;2")
smachno.add_order("2024-07-20 20:15;980.00;120;4")
smachno.add_delivery(1, "Оболонь", "D-3")

for attempt in [(1, "Поділ", "D-1"), (5, "Поділ", "D-1")]:
    try:
        smachno.add_delivery(*attempt)
    except ValueError as error:
        print("Помилка:", error)

print(smachno.orders)
print(smachno.report())

## 🛠 Вправа 3. Звіт водіїв

Додай у `DeliveryService` метод `drivers()` — словник «водій → виторг» від більшого до меншого. Клітинка нижче додає метод до вже оголошеного класу (так роблять лише в ноутбуці; у файлі — пиши метод у самому класі).

In [ ]:
from collections import Counter


def drivers(self):
    # YOUR CODE HERE
    # BEGIN SOLUTION
    totals = Counter()
    for delivery in self.deliveries.values():
        totals[delivery.driver] += delivery.fare()
    return dict(totals.most_common())
    # END SOLUTION


DeliveryService.drivers = drivers

service = DeliveryService()
for line in ["2024-07-19 18:30;540.00;50;2", "2024-07-20 20:15;980.00;120;4", "2024-07-21 12:10;320.00;30;1"]:
    service.add_order(line)
service.add_delivery(1, "Оболонь", "D-3")
service.add_delivery(2, "Оболонь", "D-3")
service.add_delivery(3, "Поділ", "D-1")
print(service.drivers())
assert service.drivers() == {"D-3": 160, "D-1": 60}
assert DeliveryService().drivers() == {}
print("✅ Вправа 3 пройдена")

## 🛠 Вправа 4. Промокод як клас

Перепиши обмежений промокод з уроку 18 класом `Promo(code, percent, uses)`: метод `apply(bill)`, атрибут `left`, `__repr__` → `Promo(LUCKY20, 20%, лишилось 2)`, `@classmethod from_text("SUMMER10:10:100")`. `percent` поза 1–100 → `ValueError` у `__init__`.

In [ ]:
class Promo:
    # YOUR CODE HERE
    # BEGIN SOLUTION
    def __init__(self, code, percent, uses):
        if not 1 <= percent <= 100:
            raise ValueError(f"знижка має бути від 1 до 100 %, а маємо {percent}")
        self.code = code
        self.percent = percent
        self.left = uses

    def __repr__(self):
        return f"Promo({self.code}, {self.percent}%, лишилось {self.left})"

    @classmethod
    def from_text(cls, text):
        code, percent, uses = text.split(":")
        return cls(code, int(percent), int(uses))

    def apply(self, bill):
        if self.left == 0:
            raise ValueError(f"промокод {self.code} вичерпано")
        self.left -= 1
        return round(bill * (100 - self.percent) / 100, 2)
    # END SOLUTION


lucky = Promo("LUCKY20", percent=20, uses=3)
assert lucky.apply(500.0) == 400.0 and lucky.left == 2
assert repr(lucky) == "Promo(LUCKY20, 20%, лишилось 2)"
lucky.apply(500.0)
lucky.apply(250.0)
try:
    lucky.apply(500.0)
except ValueError as error:
    assert str(error) == "промокод LUCKY20 вичерпано"
else:
    raise AssertionError("очікували ValueError")
summer = Promo.from_text("SUMMER10:10:100")
assert (summer.code, summer.percent, summer.left) == ("SUMMER10", 10, 100)
try:
    Promo("BAD", 150, 1)
except ValueError:
    pass
else:
    raise AssertionError("знижка 150% — ValueError")
print(summer)
print("✅ Вправа 4 пройдена")

<details>
<summary>Замикання чи клас — що зручніше перевіряти?</summary>

Клас довший, але стан видно (`lucky.left`), є зрозумілий `__repr__`, альтернативний конструктор і перевірка в одному місці. Тест може подивитися `left` напряму — у замиканні стан схований.

</details>

## ✅ Самоперевірка

1. Що відбувається при `Order(1, …)`?
2. Що таке `self`?
3. Чому `courier.trips` — 2, а `Courier.trips` — 0?
4. Чому метод не бачить `VAT` без `self.`?
5. Коли потрібен `@classmethod`, а коли `@staticmethod`?

<details>
<summary>Відповіді</summary>

1. Python створює об'єкт і викликає `__init__`, щоб записати атрибути (і перевірити дані).
2. Сам екземпляр, у якого викликали метод.
3. Присвоєння через `self` створило атрибут в екземплярі.
4. Простір імен класу не входить у LEGB методів.
5. `@classmethod` — альтернативний конструктор (`cls`); `@staticmethod` — правило без `self` і `cls`.

</details>

### Шпаргалка

```python
class Order:
    FEES = {...}                         # атрибут класу — один на всіх

    def __init__(self, order_id, bill):  # викликається при Order(...)
        self.id = order_id               # атрибути екземпляра
        self.items = []                  # змінювані дані — тільки тут!

    def __repr__(self):                  # як показувати об'єкт
        return f"Order(№{self.id})"

    def total(self):                     # метод: order.total() == Order.total(order)
        return ...

    @classmethod
    def from_line(cls, line): ...        # альтернативний конструктор

    @staticmethod
    def meal_type(hour): ...             # правило без self і cls

obj.__dict__                             # простір імен екземпляра
isinstance(obj, Order)                   # краще за type(obj) is Order
```

## Далі

- Практикум на реальних даних — [`lab_lesson_19_titanic_oop.ipynb`](https://github.com/NikoriakViktot/PY-Course-Victor-Nikoriak-22-09-2026/blob/main/module_2/lessons/lesson_19_classes_namespace/lab_lesson_19_titanic_oop.ipynb): клас `Passenger` для пасажирів «Титаніка».
- **Урок 20 — Наслідування, поліморфізм**: доставка таксі, кур'єром і самовивіз — різні класи з однаковим методом `fare()`.